In [1]:
verbose = False

In [2]:
# All the imports

import torch
from torch.utils.data import TensorDataset
from torch import nn
import matplotlib.pyplot as plt

from load_dataset import labelList, get_data
from preprocess import preprocess_dataset
import bigmodel

from ml_util.eeg_util import plotEEG, plotTimeFreqEEG
from ml_util.checkpoint import checkpoint
from ml_util.data_module import DataModule, calculate_accuracy
from ml_util.trainer import Trainer
from ml_util.report_wandb import WandBReporter, print_stats

In [3]:
X_raw, y = get_data(verbose=True)

Loading dataset


                               | 0/480 [          ]

Dataset loaded, X : torch.Size([480, 32, 3200])


In [4]:
if verbose:
    plotEEG(X_raw[0], title="index 0")
    print(labelList[y[0]])

In [5]:
# Applies preprocessing
X = checkpoint(lambda : preprocess_dataset(X_raw, verbose=True), "preprocessed")
print(f"Input dataset shape: {X.shape}")

# Stores in a TensorDataset
dataset = DataModule(X, y)

Input dataset shape: torch.Size([480, 32, 32, 51])


In [6]:
if verbose:
    for label in range(4):
        sampleId = torch.where(y==label)[0][0]
        plt.figure()
        fig, axes = plotTimeFreqEEG(X[sampleId])
        #plt.imshow(X[sampleId][0])
        fig.suptitle(f"Sample {sampleId}, with label {labelList[y[sampleId]]}")
    plt.show()

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("GPU" if torch.cuda.is_available() else "CPU")

hyperparams = {
    'lr': 4e-3,
    'weight_decay': 0,
    'batch_size': 16
}
model = bigmodel.model(X[0])
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=hyperparams['lr'], weight_decay=hyperparams['weight_decay'])

trainer = Trainer(model, optimizer, loss_fn, hyperparams, device)

def babysitter(hyperparameters, epoch_results):
    hyperparameters['lr'] = 0.9 * hyperparameters['lr']

wandbreporter = WandBReporter(hyperparams)
def report(res):
    print_stats(res)
    wandbreporter.report(res)

trainer.set_report_callback(report)
trainer.set_babysitting_callback(babysitter)

CPU


wandb: Currently logged in as: ulyssedurand (ulyssedurand-ens-de-lyon) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [8]:
trainer.train(dataset, 1000)

Epoch:    0    | tl: 25.813072, ta: 0.187500    |    vl: 248.868958, va: 0.263889
Epoch:    1    | tl: 30.138811, ta: 0.312500    |    vl: 155.498566, va: 0.291667
Epoch:    2    | tl: 32.924641, ta: 0.312500    |    vl: 133.973450, va: 0.166667
Epoch:    3    | tl: 35.095539, ta: 0.125000    |    vl: 132.584488, va: 0.138889
Epoch:    4    | tl: 28.410467, ta: 0.437500    |    vl: 110.859650, va: 0.222222
Epoch:    5    | tl: 25.992577, ta: 0.250000    |    vl: 106.351669, va: 0.236111
Epoch:    6    | tl: 25.354820, ta: 0.375000    |    vl: 102.489258, va: 0.222222
Epoch:    7    | tl: 24.500834, ta: 0.250000    |    vl: 100.969818, va: 0.291667
Epoch:    8    | tl: 23.812962, ta: 0.312500    |    vl: 99.830307, va: 0.291667
Epoch:    9    | tl: 23.314854, ta: 0.312500    |    vl: 99.643318, va: 0.291667


KeyboardInterrupt: 